# Universal Silica Fiber — 3D Structure & Propagation (v0.1)

**Research-mode 2+1D scalar split-step FFT BPM** for straight solid silica fibers (Step, Graded, Trench, Multilayer). The code propagates a complex transverse optical envelope in z and stores a 3D stack of |A(x,y,z)|². This is **not 3D Full-vector Maxwell FEM/FDTD**, HCF/NANF leakage, polarization, Rayleigh scattering or physical fiber attenuation. z-axis is compressed in the 3D plot for visibility.

## 1 · Pinned numerical code (self-contained, no GitHub runtime download)

In [ ]:
from pathlib import Path
Path('propagation_3d.py').write_text("\"\"\"Solid silica fiber scalar paraxial (2+1)D split-step FFT beam propagation.\nReal 3D intensity slices E(x,y,z) calculated by longitudinal marching.\nNOT 3D Full-vector Maxwell/FEM/FDTD. HCF/NANF leaky modes are NOT supported.\nx/y/z in micrometres; wavelength in nanometres.\n\"\"\"\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict\nimport numpy as np\n\n_B=(.6961663,.4079426,.8974794)\n_C=(.0684043**2,.1162414**2,9.896161**2)\n\ndef silica_index(wavelength_nm):\n    l=wavelength_nm/1000.\n    if not .21<l<3.7: raise ValueError(\"Silica Sellmeier outside 210–3700 nm\")\n    return float(np.sqrt(1+sum(b*l*l/(l*l-c) for b,c in zip(_B,_C))))\n\n@dataclass\nclass Design:\n    profile:str='step' # step, graded, trench, multilayer, homogeneous\n    wavelength_nm:float=1550.\n    core_radius_um:float=5.5\n    delta_n_core:float=.0034785793846155\n    graded_alpha:float=2.\n    trench_start_um:float=7.\n    trench_end_um:float=10.\n    trench_delta_n:float=-.001\n    shells:tuple=((4.,.006),(5.,.002),(7.,0.),(9.,-.0008))\n    half_width_um:float=36.\n    grid:int=128\n    length_um:float=300.\n    steps:int=120\n    waist_um:float=4.5\n    offset_um:float=1.5\n    snapshots:int=9\n\ndef index_map(d,X,Y):\n    radius=np.hypot(X,Y);n=silica_index(d.wavelength_nm)\n    if d.profile=='homogeneous':return np.full_like(radius,n)\n    if d.profile=='step':dn=np.where(radius<d.core_radius_um,d.delta_n_core,0.)\n    elif d.profile=='graded':dn=np.where(radius<d.core_radius_um,d.delta_n_core*np.maximum(1-(radius/d.core_radius_um)**d.graded_alpha,0.),0.)\n    elif d.profile=='trench':dn=np.where(radius<d.core_radius_um,d.delta_n_core,np.where((radius>=d.trench_start_um)&(radius<d.trench_end_um),d.trench_delta_n,0.))\n    elif d.profile=='multilayer':\n        dn=np.zeros_like(radius);prev=0\n        for boundary,value in d.shells:dn=np.where((radius>=prev)&(radius<boundary),value,dn);prev=boundary\n    else:raise ValueError(\"Unsupported profile; HCF geometry requires complex full-vector/PML solver\")\n    return n+dn\n\ndef metrics(E,X,Y,dx,halfwidth):\n    I=np.abs(E)**2;power=float(I.sum()*dx*dx)\n    xc=float(np.sum(X*I)*dx*dx/power);yc=float(np.sum(Y*I)*dx*dx/power)\n    variance=float(np.sum(((X-xc)**2+(Y-yc)**2)*I)*dx*dx/power)\n    aeff=float(power*power/(np.sum(I*I)*dx*dx))\n    border=(np.abs(X)>.85*halfwidth)|(np.abs(Y)>.85*halfwidth)\n    edge=float(np.sum(I[border])*dx*dx/power)\n    return dict(power=power,centroid_x_um=xc,centroid_y_um=yc,\n                MFD_2nd_um=2*np.sqrt(2*variance),Aeff_um2=aeff,edge_power_fraction=edge)\n\ndef propagate(d=Design()):\n    if d.profile not in ('step','graded','trench','multilayer','homogeneous'):\n        raise ValueError('HCF not supported by scalar BPM')\n    if not isinstance(d.grid,int) or not 32<=d.grid<=512 or d.grid&(d.grid-1):\n        raise ValueError('Grid must be power of two from 32 to 512')\n    if not isinstance(d.steps,int) or not 1<=d.steps<=2000 or not 2<=d.snapshots<=d.steps+1:\n        raise ValueError('Invalid steps/snapshots')\n    if not np.isfinite(d.half_width_um) or d.half_width_um < max(2*d.core_radius_um,4*d.waist_um):\n        raise ValueError('Enlarge periodic boundary window')\n    if not (d.length_um>0 and d.waist_um>0 and np.isfinite(d.offset_um)):\n        raise ValueError('Invalid geometry / launch')\n    if d.profile=='trench' and not(d.core_radius_um<d.trench_start_um<d.trench_end_um<d.half_width_um):\n        raise ValueError('Invalid trench dimensions')\n    if d.profile=='multilayer':\n        prev=0\n        for r,dn in d.shells:\n            if not(prev<r<d.half_width_um and np.isfinite(dn)):raise ValueError('Invalid shells')\n            prev=r\n    n=d.grid;dx=2*d.half_width_um/n\n    a=(np.arange(n)-n//2)*dx;X,Y=np.meshgrid(a,a,indexing='xy')\n    nm=index_map(d,X,Y);nref=silica_index(d.wavelength_nm);k0=2*np.pi/(d.wavelength_nm/1000)\n    dz=d.length_um/d.steps\n    q=2*np.pi*np.fft.fftfreq(n,d=dx);KX,KY=np.meshgrid(q,q,indexing='xy')\n    halfphase=np.exp(1j*dz*k0*(nm*nm-nref*nref)/(4*nref))\n    diffraction=np.exp(-1j*dz*(KX*KX+KY*KY)/(2*k0*nref))\n    E=np.exp(-((X-d.offset_um)**2+Y*Y)/d.waist_um**2).astype(complex)\n    E/=np.sqrt(np.sum(np.abs(E)**2)*dx*dx)\n    mark=set(np.rint(np.linspace(0,d.steps,d.snapshots)).astype(int))\n    history=[];snaps=[]\n    for j in range(d.steps+1):\n        row=dict(z_um=j*dz,**metrics(E,X,Y,dx,d.half_width_um));history.append(row)\n        if j in mark:snaps.append(dict(z_um=j*dz,intensity=np.abs(E)**2))\n        if j<d.steps:\n            E*=halfphase;E=np.fft.ifft2(np.fft.fft2(E)*diffraction);E*=halfphase\n    return dict(design=asdict(d),axis_um=a,index_map=nm,history=history,slices=snaps,\n                max_power_error=max(abs(m['power']-1) for m in history),\n                HCF_loss_dB_km=None,\n                solver='2+1D scalar paraxial FFT BPM; no 3D vector FEM, no PML',\n                boundary='periodic FFT; check boundary energy and convergence')\n\ndef export(r,folder='results'):\n    from pathlib import Path\n    import json,csv\n    p=Path(folder);p.mkdir(parents=True,exist_ok=True)\n    fields=list(r['history'][0]); \n    with (p/'propagation_metrics.csv').open('w',newline='') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(r['history'])\n    summary={k:v for k,v in r.items() if k not in ('axis_um','index_map','slices','history')}\n    summary['final']=r['history'][-1]\n    (p/'propagation_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))\n    np.savez_compressed(p/'field_volume_slices.npz',x=r['axis_um'],index=r['index_map'],\n          z=np.array([s['z_um'] for s in r['slices']]),intensity=np.array([s['intensity'] for s in r['slices']]))\n    return summary\n\nif __name__=='__main__':\n    r=propagate();print(export(r))\n", encoding='utf-8')
print('Wrote standalone split-step Fourier BPM source')

## 2 · Change the design, then click Runtime → Run all

In [ ]:
from propagation_3d import Design, propagate, export, silica_index
import numpy as np, matplotlib.pyplot as plt
# All lengths: µm except wavelength_nm.
design=Design(profile='step',wavelength_nm=1550,core_radius_um=5.5,delta_n_core=0.0034785793846155,
              half_width_um=36,grid=128,length_um=300,steps=120,waist_um=4.5,offset_um=1.5,snapshots=9)
print(design)

## 3 · Calculate actual propagation and integrate MFD / Aeff

In [ ]:
out=propagate(design)
first,last=out['history'][0],out['history'][-1]
print('Initial 2nd-moment MFD:',round(first['MFD_2nd_um'],4),'µm')
print('Final   2nd-moment MFD:',round(last['MFD_2nd_um'],4),'µm')
print('Final Aeff:',round(last['Aeff_um2'],4),'µm²')
print('Numerical power deviation:',out['max_power_error'])
print('Edge fraction:',last['edge_power_fraction'])
assert out['max_power_error']<1e-9

## 4 · Index map, actual propagated intensity and 3D field slices

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
axis=out['axis_um'];s=out['slices'];fig,axs=plt.subplots(1,4,figsize=(17,3.8),constrained_layout=True)
axs[0].imshow(out['index_map'],origin='lower',extent=[axis[0],axis[-1],axis[0],axis[-1]])
axs[0].set_title('n(x,y) · input geometry')
for ax,idx in zip(axs[1:],[0,len(s)//2,len(s)-1]):
    v=s[idx];ax.imshow(v['intensity'],origin='lower',cmap='magma',extent=[axis[0],axis[-1],axis[0],axis[-1]])
    ax.set_title('BPM intensity z=%.1f µm'%v['z_um'])
for ax in axs:ax.set(xlabel='x (µm)',ylabel='y (µm)')
plt.show()
fig=plt.figure(figsize=(8,5));ax=fig.add_subplot(111,projection='3d');step=max(1,len(axis)//21)
X,Y=np.meshgrid(axis[::step],axis[::step]);vmax=max(float(snap['intensity'].max()) for snap in s)
for snap in s[::2]:
    norm=snap['intensity'][::step,::step]/vmax;rgba=plt.get_cmap('magma')(norm);rgba[...,3]=np.clip(norm*1.6,.06,.8)
    ax.plot_surface(X,Y,np.full_like(X,snap['z_um']),facecolors=rgba,linewidth=0,shade=False)
ax.set(xlabel='x µm',ylabel='y µm',zlabel='z µm',title='Real 2+1D BPM slices (not 3D Maxwell FEM)');ax.set_box_aspect((1,1,2.3));plt.show()

## 5 · Propagation diagnostics and Gaussian analytic oracle

In [ ]:
history=out['history'];plt.figure(figsize=(8,3.8));plt.plot([h['z_um'] for h in history],[h['MFD_2nd_um'] for h in history]);plt.xlabel('z (µm)');plt.ylabel('MFD 2nd moment (µm)');plt.grid(alpha=.25);plt.show()
from dataclasses import replace
reference=Design(profile='homogeneous',grid=128,half_width_um=45,length_um=90,steps=90,waist_um=6,offset_um=0,snapshots=5)
ref=propagate(reference);zR=np.pi*silica_index(reference.wavelength_nm)*reference.waist_um**2/(reference.wavelength_nm/1000)
expected=2*reference.waist_um*np.sqrt(1+(reference.length_um/zR)**2)
actual=ref['history'][-1]['MFD_2nd_um'];print('Homogeneous analytic MFD',expected,'numerical MFD',actual)
assert abs(actual/expected-1)<.015

## 6 · Export CSV / JSON / compressed field volume

In [ ]:
summary=export(out,'results')
from pathlib import Path
import zipfile
with zipfile.ZipFile('Fiber_3D_BPM_Results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in Path('results').iterdir(): z.write(f,f.name)
print('Saved results/ and Fiber_3D_BPM_Results.zip')
# Optional in Colab:
# from google.colab import files
# files.download('Fiber_3D_BPM_Results.zip')

## Limits

- Gaussian input is **not automatically a guided eigenmode**. MFD and Aeff vary with z and should not be mistaken for manufacturer LP01 modal specifications.
- Real-valued index and FFT periodic boundaries conserve numerical power, so **power conservation ≠ measured attenuation**.
- The HCF nested-tube 3D geometry is viewed separately in the web UI. No HCF full-vector propagation or confinement-loss dB/km prediction.
- Check transverse domain, x/y grid and z step convergence. Full 3D vector Maxwell and radiation/PML will require a separate solver.